# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'
ratings = ['TV-MA', 'TV-14', 'PG-13', 'R', 'PG']
pivot = (df[df['rating'].isin(ratings)]
         .groupby(['rating', 'decade']).size()
         .unstack(fill_value=0).reindex(ratings))

fig = px.imshow(pivot, color_continuous_scale='Blues', text_auto=True,
                labels={'color': 'Titles'}, height=500, width=700,
                title='TV-MA leads every decade, but PG-13 fades fastest')
fig.update_yaxes(title='Rating')
fig.update_xaxes(title='Release decade')
fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [4]:
movies = df[df['type'] == 'Movie']
adds = (movies[movies['added_year'].between(2015, 2022)]
        .groupby('added_year').size())
cum = adds.cumsum()
peak = adds.idxmax()
total = int(adds.sum())
labels = adds.index.astype(str).tolist() + ['Total']

fig = go.Figure(go.Waterfall(
    x=labels,
    y=adds.tolist() + [total],
    measure=['relative'] * len(adds) + ['total'],
    text=[f'+{v}' for v in adds] + [f'{total}'],
    textposition='outside',
    increasing=dict(marker_color='#70AD47'),
    totals=dict(marker_color='#2E75B6'),
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
))

fig.add_annotation(x=str(peak), y=int(cum.loc[peak]),
                   text=f'Peak intake: {int(adds.max())} films in {peak}',
                   showarrow=True, arrowhead=2, ax=40, ay=-40,
                   bgcolor='#FFF7CC', bordercolor='#888')

fig.update_layout(
    title='Netflix added ~80 movies a year, peaking in 2016 — library grew to 659 by 2022',
    plot_bgcolor='white', height=500,
    showlegend=False,
    margin=dict(l=60, r=30, t=70, b=50),
    bargap=0.2,
    xaxis=dict(title='Year', type='category',
               categoryorder='array', categoryarray=labels,
               range=[-0.5, len(labels) - 0.5]),
    yaxis=dict(title='Movies added (cumulative)', gridcolor='#EEEEEE'))
fig.show()